# 🧊 EvalSpace — Verifiable Spatial Reasoning Environments

Physics-verified environments for evaluating and training VLMs on spatial reasoning.

**What this notebook shows:**
1. Install EvalSpace from GitHub
2. Create an environment and generate scenes
3. Inspect images, questions, and ground truth
4. Verify model predictions
5. Generate a dataset at scale
6. Evaluate a frontier model (OpenRouter)

## 1. Install

In [ ]:
!pip install -q mujoco Pillow numpy httpx
!pip install -q git+https://github.com/abishek21/EvalSpace.git

## 2. Create Environment

In [ ]:
import evalspace as es

print(f"EvalSpace v{es.__version__}")

# Create a shelf fitting environment
env = es.make(
    task="shelf_fitting",
    difficulty="medium",   # easy | medium | hard
    seed=42,
)
print(f"Environment: {type(env).__name__}")

## 3. Generate a Scene

In [ ]:
# Generate a new scene
obs = env.reset()

# Display the image
from IPython.display import display
display(obs.image)

# Print the question
print(f"\nQuestion: {obs.question}")
print(f"\nGround Truth: {env.ground_truth()}")

## 4. Inspect Scene Metadata

All spatial information is in the image — questions contain NO dimension hints.

In [ ]:
meta = obs.metadata

print("Target object:")
t = meta["target"]
print(f"  {t['color']} {t['label']}: {t['width']*100:.0f}cm W × {t['depth']*100:.0f}cm D × {t['height']*100:.0f}cm H")

print(f"\nShelf:")
s = meta["shelf"]
print(f"  Clearance: {s['clearance']*100:.0f}cm, Depth: {s['depth']*100:.0f}cm, Width: {s['width']*100:.0f}cm")

print(f"\nExisting objects on shelf: {meta['num_existing_objects']}")
print(f"Difficulty: {meta['difficulty']}")

## 5. Verify Predictions

The verifier accepts natural language answers and returns a reward signal.

In [ ]:
gt = env.ground_truth()
print(f"Ground truth: {gt['answer']}")
print(f"Reasoning: {gt['reasoning']}")
print()

# Verify different answers
for answer in ["fits", "no_fit", "yes", "doesn't fit", "it fits", "too big"]:
    reward = env.verify(answer)
    print(f"  verify('{answer}') → reward = {reward}")

## 6. Gym-Compatible Interface

Standard `reset() / step()` for RL training loops.

In [ ]:
obs = env.reset()

# Simulate a model prediction
model_answer = "fits"  # pretend the model said this

obs, reward, done, info = env.step(model_answer)
print(f"Action: '{model_answer}'")
print(f"Reward: {reward}")
print(f"Done: {done}")
print(f"Correct: {info['correct']}")
print(f"Ground truth was: {info['ground_truth']['answer']}")

## 7. Generate Multiple Scenes

Each `reset()` produces a unique scene — infinite variety, no memorization.

In [ ]:
import matplotlib.pyplot as plt

env = es.make(task="shelf_fitting", difficulty="hard", seed=100)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, ax in enumerate(axes.flat):
    obs = env.reset()
    gt = env.ground_truth()
    ax.imshow(obs.image)
    color = "green" if gt["answer"] == "fits" else "red"
    ax.set_title(f"{gt['answer'].upper()}\n{obs.question[:50]}...", fontsize=9, color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Multi-View Rendering

Get multiple camera angles for richer visual input.

In [ ]:
env_mv = es.make(
    task="shelf_fitting",
    difficulty="hard",
    seed=55,
    multi_view=True,
    views=[
        {"azimuth": 270, "elevation": -15},   # front
        {"azimuth": 310, "elevation": -20},   # angled right
    ]
)

obs = env_mv.reset()
images = obs.metadata["images"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.imshow(images[0]); ax1.set_title("Front View"); ax1.axis("off")
ax2.imshow(images[1]); ax2.set_title("Angled View"); ax2.axis("off")
plt.suptitle(obs.question, fontsize=11)
plt.tight_layout()
plt.show()

gt = env_mv.ground_truth()
print(f"GT: {gt['answer']} — {gt['reasoning']}")

## 9. Physics Engine Verification

Use MuJoCo physics simulation instead of rule-based checks.
The physics engine drops the object on the shelf and checks if it stays.

In [ ]:
env_physics = es.make(
    task="shelf_fitting",
    difficulty="medium",
    seed=42,
    physics_engine=True,  # MuJoCo simulation verification
)

obs = env_physics.reset()
gt = env_physics.ground_truth()

display(obs.image)
print(f"Question: {obs.question}")
print(f"GT: {gt['answer']}")
print(f"Reasoning: {gt['reasoning']}")

## 10. Generate a Dataset

Create a static dataset and save locally (or push to HuggingFace).

In [ ]:
# Generate 20 scenes
suite = es.generate(
    task="shelf_fitting",
    num_scenes=20,
    difficulty="mixed",
    seed=123,
)
print(suite)

# Save locally
suite.save("./evalspace_demo_dataset")

# Push to HuggingFace (uncomment and add your token)
# suite.push_to_hub("your-org/shelf-fitting-20", token="hf_...")

## 11. Evaluate a Model (OpenRouter)

Run a frontier model against the dataset and get accuracy scores.

In [ ]:
# Generate a small test suite
test_suite = es.generate(task="shelf_fitting", num_scenes=5, seed=42)

# Evaluate (uncomment and add your API key)
# results = es.evaluate(
#     test_suite,
#     model="openrouter/x-ai/grok-4.3",
#     api_key="sk-or-...",
# )
# results.summary()
# results.save("./eval_results.json")

print("Uncomment above and add your OpenRouter API key to run evaluation")

## 12. RL Training Loop (Example)

How you'd use EvalSpace in an RL training loop.

In [ ]:
import random

env = es.make(task="shelf_fitting", difficulty="medium", seed=0)

# Simulate a training loop with a random "policy"
total_reward = 0
for episode in range(20):
    obs = env.reset()
    
    # Random policy (replace with your VLM)
    action = random.choice(["fits", "no_fit"])
    
    obs, reward, done, info = env.step(action)
    total_reward += reward

accuracy = total_reward / 20 * 100
print(f"Random policy accuracy: {accuracy:.0f}% (expected ~50%)")
print(f"A good VLM should score 70-90%+")

---

## Summary

| Feature | API |
|---------|-----|
| Create environment | `es.make(task, difficulty, seed)` |
| Generate scene | `env.reset()` |
| Get question | `obs.question` |
| Get image | `obs.image` |
| Verify answer | `env.verify(answer)` |
| Gym step | `env.step(action)` |
| Ground truth | `env.ground_truth()` |
| Generate dataset | `es.generate(num_scenes=5000)` |
| Save dataset | `suite.save(path)` |
| Push to HF | `suite.push_to_hub(repo_id)` |
| Evaluate model | `es.evaluate(suite, model, api_key)` |
| Physics verification | `physics_engine=True` |
| Multi-view | `multi_view=True` |